# 🤖 AutoML Pilot - Pro Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset with automated reporting and EDA.

In [ ]:
# @title 1. Install Dependencies
!pip install -q pycaret[full] ydata-profiling pandas jinja2
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f'✅ Loaded {filename}: {df.shape[0]} rows')

In [ ]:
# @title 3. Automated EDA
run_eda = True # @param {type:"boolean"}
if run_eda:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df, title="Profiling Report", explorative=True)
    profile.to_notebook_iframe()

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize

target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = cls_compare()
    leaderboard = cls_pull()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
    best_metrics = leaderboard.iloc[0].to_dict()
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = reg_compare()
    leaderboard = reg_pull()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')
    best_metrics = leaderboard.iloc[0].to_dict()

display(leaderboard.head())

In [ ]:
# @title 5. Automated Professional Email Reporting
import smtplib, ssl
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import json

recipient_email = '' # @param {type:"string"}
sender_email = '' # @param {type:"string"}
sender_password = '' # @param {type:"string"}

if recipient_email and sender_email and sender_password:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"🚀 AutoML Pilot Pro - {task_type.capitalize()} Training Report"
    msg["From"] = f"AutoML Pilot Colab <{sender_email}>"
    msg["To"] = recipient_email

    model_name = str(best_model).split('(')[0]
    metric_html = ""
    if task_type == 'classification':
        metric_html = f"<p style='margin:4px 0;'><strong>Accuracy:</strong> {best_metrics.get('Accuracy', 0)*100:.2f}%</p>"
        metric_html += f"<p style='margin:4px 0;'><strong>F1 Score:</strong> {best_metrics.get('F1', 0):.4f}</p>"
    else:
        metric_html = f"<p style='margin:4px 0;'><strong>R² Score:</strong> {best_metrics.get('R2', 0):.4f}</p>"
        metric_html += f"<p style='margin:4px 0;'><strong>RMSE:</strong> {best_metrics.get('RMSE', 0):.4f}</p>"

    html_body = f"""
    <html>
    <body style='font-family:Inter,system-ui,sans-serif; padding:20px; background:#f0f2f6;'>
      <div style='max-width:600px; margin:0 auto; background:white; padding:32px; border-radius:8px; border: 1px solid #e1e4e8;'>
        <div style='text-align:center; margin-bottom:24px;'>
            <h1 style='color:#7c3aed; margin:0; font-size:28px;'>🚀 AutoMLPilot Pro</h1>
            <p style='color:#6b7280; margin:4px 0 0 0;'>Colab Training Report</p>
        </div>

        <div style='background:#f9fafb; padding:20px; border-radius:12px; margin-bottom:24px; border-left: 4px solid #7c3aed;'>
            <h3 style='margin-top:0; color:#1f2937;'>📊 Best Model Summary</h3>
            <p style='margin:4px 0;'><strong>Model:</strong> {model_name}</p>
            <p style='margin:4px 0;'><strong>Task:</strong> {task_type.capitalize()}</p>
            {metric_html}
        </div>

        <h3 style='color:#1f2937; margin-bottom:12px;'>🏆 Top Models Leaderboard</h3>
        <div style='overflow-x:auto;'>
            {leaderboard.head().to_html(classes='table', border=0)}
        </div>

        <div style='text-align:center; margin-top:32px; padding-top:24px; border-top:1px solid #e1e4e8;'>
            <p style='color:#94a3b8; font-size:12px; margin:0;'>
              This report was generated automatically via AutoMLPilot Pro Colab Integration.<br>
              &copy; 2024 AutoMLPilot AI Labs
            </p>
        </div>
      </div>
      <style>
        .table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
        .table th {{ background: #f3f4f6; padding: 8px; text-align: left; border-bottom: 1px solid #e5e7eb; }}
        .table td {{ padding: 8px; border-bottom: 1px solid #f3f4f6; }}
      </style>
    </body>
    </html>
    """
    msg.attach(MIMEText(html_body, "html"))
    context = ssl.create_default_context()
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, recipient_email, msg.as_string())
    print('✅ Professional email report sent!')

In [ ]:
# @title 6. Download Model
from google.colab import files
if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
    print('✅ Model downloaded!')